In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pathlib import Path

In [5]:
train_csv = Path("~/Workspace/ML_Project/data/processed/train.csv")
val_csv = Path("~/Workspace/ML_Project/data/processed/val.csv")
test_csv = Path("~/Workspace/ML_Project/data/processed/test.csv")

movies_csv = Path("~/Workspace/ML_Project/data/row/ml-latest-small/movies.csv")

train = pd.read_csv(train_csv)
val = pd.read_csv(val_csv)
test = pd.read_csv(test_csv)
movies = pd.read_csv(movies_csv)

print(train.shape, "\n")
print(val.shape, "\n")
print(test.shape, "\n")

print(train.head(5), "\n")
print(val.head(5), "\n")
print(test.head(5), "\n")

(60419, 7) 

(7545, 7) 

(7545, 7) 

   uid   iid  userId  movieId  rating  like  timestamp
0    0   451       1     1210     5.0     1  964980499
1    0   722       1     2018     5.0     1  964980523
2    0   904       1     2628     4.0     0  964980523
3    0   961       1     2826     4.0     0  964980523
4    0  1143       1     3578     5.0     1  964980668 

   uid  iid  userId  movieId  rating  like  timestamp
0    0  360       1      954     5.0     1  964983219
1    0  449       1     1208     4.0     0  964983250
2    0  813       1     2329     5.0     1  964983263
3    0  992       1     2959     5.0     1  964983282
4    0  458       1     1219     2.0     0  964983393 

   uid   iid  userId  movieId  rating  like  timestamp
0    0   800       1     2291     5.0     1  964983664
1    0   500       1     1270     5.0     1  964983705
2    0   473       1     1240     5.0     1  964983723
3    0   447       1     1206     5.0     1  964983737
4    0  1168       1     3702 

In [6]:
import json

import torch
import torch.nn as nn

SEED = 42
K = 10
N_NEG = 4
BATCH = 1024
EPOCHS = 40
PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-5

D = Path("~/Workspace/ML_Project/data/processed").expanduser()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
np.random.seed(SEED)
torch.manual_seed(SEED)

meta = json.load(open(D / "meta.json"))
n_users, n_items = meta["n_users"], meta["n_items"]
val_neg, test_neg = np.load(D / "val_neg.npy"), np.load(D / "test_neg.npy")
print(device, n_users, n_items, val_neg.shape, test_neg.shape)

ModuleNotFoundError: No module named 'torch'

In [ ]:
def sampled_metrics(S, df, neg, k=K):
    u = df["uid"].to_numpy()
    pos = S[u, df["iid"].to_numpy()]
    negs = S[u[:, None], neg]
    rank = (negs > pos[:, None]).sum(1) + 0.5 * (negs == pos[:, None]).sum(1)
    hit = (rank < k).astype(float)
    return hit.mean(), (hit / np.log2(rank + 2)).mean()


def full_metrics(S, hist, df, k=K):
    S = np.where(hist, -np.inf, S.astype(np.float64) + np.random.default_rng(SEED).random(S.shape) * 1e-9)
    top = np.argsort(-S, axis=1)[:, :k]
    rel = np.zeros((n_users, n_items), dtype=bool)
    rel[df["uid"].to_numpy(), df["iid"].to_numpy()] = True
    gains = np.take_along_axis(rel, top, 1)
    n_rel = rel.sum(1)
    m = n_rel > 0
    disc = 1 / np.log2(np.arange(2, k + 2))
    idcg = np.array([disc[:min(int(n), k)].sum() for n in n_rel])
    recall = (gains.sum(1)[m] / n_rel[m]).mean()
    ndcg = ((gains * disc).sum(1)[m] / idcg[m]).mean()
    return recall, ndcg, len(np.unique(top[m])) / n_items


def evaluate(S, hist, df, neg):
    hr, ndcg_s = sampled_metrics(S, df, neg)
    rec, ndcg, cov = full_metrics(S, hist, df)
    return {"HR@10_100": hr, "NDCG@10_100": ndcg_s, "Recall@10": rec, "NDCG@10": ndcg, "Coverage": cov}


def to_hist(*dfs):
    h = np.zeros((n_users, n_items), dtype=bool)
    for d in dfs:
        h[d["uid"].to_numpy(), d["iid"].to_numpy()] = True
    return h


hist_train = to_hist(train)
hist_trainval = to_hist(train, val)
hist_all = to_hist(train, val, test)

In [ ]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, gmf_dim=32, mlp_dim=32, layers=(64, 32, 16), dropout=0.2):
        super().__init__()
        self.gmf_u = nn.Embedding(n_users, gmf_dim)
        self.gmf_i = nn.Embedding(n_items, gmf_dim)
        self.mlp_u = nn.Embedding(n_users, mlp_dim)
        self.mlp_i = nn.Embedding(n_items, mlp_dim)
        for e in (self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i):
            nn.init.normal_(e.weight, std=0.1)

        dims = [2 * mlp_dim, *layers]
        blocks = []
        for a, b in zip(dims[:-1], dims[1:]):
            blocks += [nn.Linear(a, b), nn.ReLU(), nn.Dropout(dropout)]
        self.mlp = nn.Sequential(*blocks)
        self.out = nn.Linear(gmf_dim + layers[-1], 1)

    def forward(self, u, i):
        g = self.gmf_u(u) * self.gmf_i(i)
        m = self.mlp(torch.cat([self.mlp_u(u), self.mlp_i(i)], dim=-1))
        return self.out(torch.cat([g, m], dim=-1)).squeeze(-1)


@torch.no_grad()
def score_all(model, chunk=64):
    model.eval()
    all_items = torch.arange(n_items, device=device)
    rows = []
    for s in range(0, n_users, chunk):
        us = torch.arange(s, min(s + chunk, n_users), device=device)
        uu = us[:, None].expand(-1, n_items).reshape(-1)
        ii = all_items[None, :].expand(len(us), -1).reshape(-1)
        rows.append(model(uu, ii).view(len(us), n_items).float().cpu())
    return torch.cat(rows).numpy()

In [ ]:
tr_u, tr_i = train["uid"].to_numpy(), train["iid"].to_numpy()
pos_keys = np.unique(tr_u.astype(np.int64) * n_items + tr_i)
rng = np.random.default_rng(SEED)


def sample_negatives(n_neg):
    u = np.repeat(tr_u, n_neg)
    neg = rng.integers(0, n_items, len(u))
    while True:
        bad = np.isin(u.astype(np.int64) * n_items + neg, pos_keys)
        if not bad.any():
            return u, neg
        neg[bad] = rng.integers(0, n_items, bad.sum())

In [ ]:
model = NeuMF(n_users, n_items).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.BCEWithLogitsLoss()

log, best_ndcg, best_epoch, best_state, bad_epochs = [], -1.0, 0, None, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    nu, ni = sample_negatives(N_NEG)
    U = torch.from_numpy(np.concatenate([tr_u, nu])).long().to(device)
    I = torch.from_numpy(np.concatenate([tr_i, ni])).long().to(device)
    Y = torch.cat([torch.ones(len(tr_u)), torch.zeros(len(nu))]).to(device)
    perm = torch.randperm(len(U)).to(device)

    total = 0.0
    for s in range(0, len(U), BATCH):
        idx = perm[s:s + BATCH]
        opt.zero_grad()
        loss = loss_fn(model(U[idx], I[idx]), Y[idx])
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)

    m = evaluate(score_all(model), hist_train, val, val_neg)
    log.append({"epoch": epoch, "loss": total / len(U), **m})
    print(f"{epoch:2d} | loss {total / len(U):.4f} | HR@10 {m['HR@10_100']:.4f} | "
          f"NDCG@10 {m['NDCG@10_100']:.4f} | full NDCG@10 {m['NDCG@10']:.4f}")

    if m["NDCG@10_100"] > best_ndcg:
        best_ndcg, best_epoch, bad_epochs = m["NDCG@10_100"], epoch, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            break

model.load_state_dict(best_state)
print(f"Лучшая эпоха: {best_epoch}, val NDCG@10 (100 кандидатов) = {best_ndcg:.4f}")

In [ ]:
log_df = pd.DataFrame(log)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log_df["epoch"], log_df["loss"])
ax[0].set(title="Train loss", xlabel="Эпоха")
ax[1].plot(log_df["epoch"], log_df["HR@10_100"], label="HR@10")
ax[1].plot(log_df["epoch"], log_df["NDCG@10_100"], label="NDCG@10")
ax[1].axvline(best_epoch, color="red", ls="--", label=f"лучшая эпоха {best_epoch}")
ax[1].set(title="Val, 100 кандидатов", xlabel="Эпоха")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
S_ncf = score_all(model)
S_pop = np.tile(hist_train.sum(0).astype(np.float64), (n_users, 1))

res = pd.DataFrame({
    "NCF val": evaluate(S_ncf, hist_train, val, val_neg),
    "Popularity val": evaluate(S_pop, hist_train, val, val_neg),
    "NCF test": evaluate(S_ncf, hist_trainval, test, test_neg),
    "Popularity test": evaluate(S_pop, hist_trainval, test, test_neg),
}).T.round(4)
res

In [ ]:
item_info = (pd.concat([train, val, test])[["iid", "movieId"]].drop_duplicates()
               .merge(movies[["movieId", "title", "genres"]], on="movieId")
               .set_index("iid").sort_index())
uid_of = dict(zip(train["userId"], train["uid"]))
all_df = pd.concat([train, val, test])


def recommend(user_id, n=10):
    u = uid_of[user_id]
    s = S_ncf[u].astype(np.float64).copy()
    s[hist_all[u]] = -np.inf
    top = np.argsort(-s)[:n]
    out = item_info.loc[top, ["title", "genres"]].copy()
    out["prob"] = 1 / (1 + np.exp(-s[top]))
    return out.reset_index()


def history(user_id, n=10):
    h = all_df[all_df["userId"] == user_id].sort_values("rating", ascending=False).head(n)
    return h.merge(item_info, left_on="iid", right_index=True)[["title", "genres", "rating"]]


demo_user = train["userId"].iloc[0]
print(history(demo_user).to_string(index=False), "\n")
print(recommend(demo_user).to_string(index=False))